# Arquitectura backend + base de experimentación (issue #12)
Usa los **mismos casos de uso** que la API (`backend/src/collection/`).
Backend **pelado por decisión (S8)**: solo plantillas, firmas y contratos abstractos, sin implementaciones.
Este notebook verifica estructura y fixtures, no entrena ni planifica (eso queda para la reimplementación).

In [ ]:
import sys
from pathlib import Path

# Funciona desde la raíz del repo y desde notebooks/ (nbval ejecuta
# el kernel con cwd = directorio del notebook).
CANDIDATOS = [Path.cwd() / "backend" / "src",
              Path.cwd().parent / "backend" / "src"]
SRC = next(c for c in CANDIDATOS if (c / "collection").is_dir())
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import collection
print("src:", SRC.exists(), "schema:", collection.SCHEMA_VERSION)

## Parámetros (el único lugar que hay que editar)

In [ ]:
# Cambiar dataset SOLO acá.
# SRC viene de la celda de setup; el default apunta al fixture local.
DATASET_DIR = str((SRC.parent / "tests" / "fixtures") if (SRC.parent / "tests" / "fixtures").is_dir() else (SRC / ".." / "tests" / "fixtures"))
print("dataset:", DATASET_DIR)

## 1. Estructura pelada (contratos abstractos, sin lógica)

In [ ]:
from abc import ABC

from collection.adapters.datasets.csv_dataset_reader import CsvDatasetReader
from collection.adapters.graphs.networkx_route_planner import NetworkxRoutePlanner
from collection.adapters.models.rule_v1_predictor import RuleV1Predictor
from collection.adapters.models.sklearn_evaluator import SklearnEvaluator
from collection.adapters.models.sklearn_predictor import SklearnPredictor
from collection.application.use_cases.compare_models.contract import CompareModelsContract
from collection.application.use_cases.plan_route.contract import PlanRouteContract
from collection.application.use_cases.predict_priorities.contract import PredictPrioritiesContract
from collection.application.use_cases.validate_load_dataset.contract import ValidateLoadDatasetContract
from collection.domain.entities.node import Node
from collection.domain.entities.predicted_segment import PredictedSegment
from collection.domain.entities.route import Route
from collection.domain.entities.route_config import RouteConfig
from collection.domain.entities.segment_features import SegmentFeatures

assert collection.SCHEMA_VERSION == "v1", "schema inesperado"
for contract in (ValidateLoadDatasetContract, CompareModelsContract, PredictPrioritiesContract, PlanRouteContract):
    assert issubclass(contract, ABC), contract
    assert "execute" in contract.__abstractmethods__, contract

assert CsvDatasetReader is not None
assert NetworkxRoutePlanner is not None
assert RuleV1Predictor is not None
assert SklearnEvaluator is not None
assert SklearnPredictor is not None
print("contratos ABC ok + adaptadores presentes (plantillas)")

## 2. Fixtures locales (esquema v1, sin pandas)

In [ ]:
import csv
from pathlib import Path

base = Path(DATASET_DIR)
nodos_p = base / "nodos.csv"
tramos_p = base / "tramos.csv"
assert nodos_p.is_file(), f"falta {nodos_p}"
assert tramos_p.is_file(), f"falta {tramos_p}"

with open(nodos_p, newline="", encoding="utf-8") as f:
    nodos = list(csv.DictReader(f))
with open(tramos_p, newline="", encoding="utf-8") as f:
    tramos = list(csv.DictReader(f))

print(f"nodos={len(nodos)} tramos={len(tramos)}")
assert len(nodos) == 6, f"fixture nodos cambió: {len(nodos)}"
assert len(tramos) == 12, f"fixture tramos cambió: {len(tramos)}"
assert {r["prioridad_recoleccion"] for r in tramos} <= {"baja", "media", "alta"}
print("ejemplo:", tramos[0]["tramo_id"], tramos[0]["prioridad_recoleccion"])

## 3. Composición pelada (los stubs levantan NotImplementedError por diseño)

In [ ]:
from collection import composition

for nombre in ("create_validate_load_dataset_use_case", "create_compare_models_use_case",
               "create_plan_route_use_case"):
    fn = getattr(composition, nombre)
    try:
        fn()
    except NotImplementedError:
        print(nombre, "-> NotImplementedError (esperado)")
    else:
        raise AssertionError(f"{nombre} debería ser plantilla sin implementar")

try:
    composition.create_predict_priorities_use_case("arbol", [], seed=0)
except NotImplementedError:
    print("create_predict_priorities_use_case -> NotImplementedError (esperado)")
else:
    raise AssertionError("predict debería ser plantilla sin implementar")

# Reglas de dominio también son plantillas (S8):
from collection.domain.rules.label_v1 import label_v1
try:
    label_v1(2, 0, 1, "residential")
except NotImplementedError:
    print("label_v1 -> NotImplementedError (esperado, pendiente #10)")
else:
    raise AssertionError("label_v1 debería ser plantilla")

> **Aviso:** la etiqueta es *construida* con la regla v1: el modelo aprenderá esa regla y una buena métrica **no** demostrará utilidad operativa real. La comparación/entrenamiento y la planificación (Hierholzer / cartero chino) se reimplementan en el issue #10 sobre estos mismos contratos.